In [44]:
import os
import json
import glob
import gc
import shutil
import collections.abc
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import darts
from darts import TimeSeries
from darts.models import TFTModel
from darts.utils.likelihood_models import NegativeBinomialLikelihood
from pytorch_lightning import Callback
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

from tft_checks import embedding_report, assert_embeddings, EmbeddingGate

print("darts", darts.__version__, "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

darts 0.40.0 | torch 2.2.2 | CUDA: True


In [30]:
BASE = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in October\_1_Data preparation"
with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)

time_col, group_col, target_col = ROLES["time_col"], ROLES["group_col"], ROLES["target_col"]
FREQ              = ROLES["freq"]
static_covariates = ROLES["static_covariates"]
future_covariates = ROLES["future_covariates"]

TRAIN_START    = pd.Timestamp(ROLES["train_start"])
TRAIN_END      = pd.Timestamp(ROLES["train_end"])
VAL_START      = pd.Timestamp(ROLES["val_start"])
VAL_END        = pd.Timestamp(ROLES["val_end"])
FORECAST_START = pd.Timestamp(ROLES["forecast_start"])
FORECAST_END   = pd.Timestamp(ROLES["forecast_end"])

INPUT_CHUNK_LENGTH  = ROLES["input_chunk_length"]      # 365
OUTPUT_CHUNK_LENGTH = ROLES["output_chunk_length"]     # 101
MIN_LEN = INPUT_CHUNK_LENGTH + OUTPUT_CHUNK_LENGTH      # shortest series that yields one training window

local_train_dir = os.path.join(BASE, "local_train_data")
CACHE_DIR       = os.path.join(BASE, "series_cache")

# festive sample weighting: weight = 1 + FESTIVE_WEIGHT on any day where a festive-window column is non-zero
FESTIVE_WEIGHT = 3.0
PENALTY_COLS = ([f"N-{i}" for i in range(16, 0, -1)] + ["N"] + [f"N+{i}" for i in range(1, 11)]
                + [f"D-{i}" for i in range(3, 0, -1)] + ["D"] + [f"D+{i}" for i in range(1, 7)]
                + ["C"] + [f"C+{i}" for i in range(1, 7)])
PENALTY_COLS = [c for c in PENALTY_COLS if c in future_covariates]

def safe_name(key):
    return str(key).replace("<>", "_").replace("/", "_").replace("\\", "_")

val_scored_start = VAL_START + pd.Timedelta(days=INPUT_CHUNK_LENGTH)
print(f"TRAIN targets      : {TRAIN_START.date()} -> {TRAIN_END.date()}")
print(f"VAL series         : {VAL_START.date()} -> {VAL_END.date()} "
      f"({(VAL_END - VAL_START).days + 1} days = {INPUT_CHUNK_LENGTH} input + {OUTPUT_CHUNK_LENGTH} output "
      f"+ {(VAL_END - VAL_START).days + 1 - MIN_LEN} extra -> "
      f"{(VAL_END - VAL_START).days + 2 - MIN_LEN} windows per series)")
print(f"VAL scored dates   : {val_scored_start.date()} -> {VAL_END.date()}  "
      f"(after TRAIN_END: {val_scored_start > TRAIN_END})")
print(f"FORECAST           : {FORECAST_START.date()} -> {FORECAST_END.date()} ({OUTPUT_CHUNK_LENGTH} days)")
print(f"festive-weighted columns: {len(PENALTY_COLS)}")

TRAIN targets      : 2023-04-01 -> 2026-04-30
VAL series         : 2025-05-01 -> 2026-08-31 (488 days = 365 input + 101 output + 22 extra -> 23 windows per series)
VAL scored dates   : 2026-05-01 -> 2026-08-31  (after TRAIN_END: True)
FORECAST           : 2026-09-01 -> 2026-12-10 (101 days)
festive-weighted columns: 37


In [31]:
with open(os.path.join(BASE, ROLES["vocab_path"])) as f:
    VOCAB = json.load(f)
assert list(VOCAB) == static_covariates, "Vocabulary columns differ from the static covariates."
assert all(v[0] == "__UNKNOWN__" for v in VOCAB.values())

def default_dim(n_categories):
    return int(min(16, max(2, round(1.6 * n_categories ** 0.56))))

EMBEDDING_DIM_OVERRIDES = {}          # e.g. {"PARENT_DEALER_CODE": 8}

EMBEDDING_SIZES = {
    c: (len(VOCAB[c]), EMBEDDING_DIM_OVERRIDES.get(c, default_dim(len(VOCAB[c]) - 1)))
    for c in static_covariates
}
print(pd.DataFrame([dict(column=c, categories=len(VOCAB[c]) - 1, table_rows=EMBEDDING_SIZES[c][0],
                         vector_size=EMBEDDING_SIZES[c][1]) for c in static_covariates]).to_string(index=False))

            column  categories  table_rows  vector_size
PARENT_DEALER_CODE        1102        1103           16
      MODEL_FAMILY          13          14            7
        BRAKE_TYPE           2           3            2
     IGNITION_TYPE           2           3            2
        WHEEL_TYPE           4           5            3
            COLOUR          27          28           10
   X_CITY_CATEGORY           3           4            3
 ZONAL_OFFICE_NAME           5           6            4
       DEALER_CITY         826         827           16


In [32]:
cal = pd.read_parquet(os.path.join(BASE, ROLES["calendar_path"]))
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.sort_values(time_col).reset_index(drop=True)
assert cal[time_col].min() <= TRAIN_START and cal[time_col].max() >= FORECAST_END, \
    "Calendar does not span TRAIN_START -> FORECAST_END."

SHARED_COV = TimeSeries.from_dataframe(cal, time_col=time_col, value_cols=future_covariates,
                                       freq=FREQ).astype(np.float32)

_flag = (cal.set_index(time_col)[PENALTY_COLS] != 0).any(axis=1)
WEIGHT_BY_DATE = (1.0 + FESTIVE_WEIGHT * _flag).astype(np.float32)
print(f"Calendar {cal[time_col].min().date()} -> {cal[time_col].max().date()}, "
      f"{len(future_covariates)} covariates; festive-weighted days: {int(_flag.sum())}")

Calendar 2023-04-01 -> 2026-12-10, 57 covariates; festive-weighted days: 144


In [34]:
REBUILD_CACHE = True

series_table_path = os.path.join(CACHE_DIR, "series_table.parquet")
RECENT_START = TRAIN_END + pd.Timedelta(days=1)   # months in history but never a training target

if not REBUILD_CACHE:
    print("Reusing existing cache (REBUILD_CACHE = False).")
    series_table = pd.read_parquet(series_table_path)
else:
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    os.makedirs(CACHE_DIR)
    rows = []
    chunk_files = sorted(glob.glob(os.path.join(local_train_dir, "chunk_*.parquet")))
    assert chunk_files, f"No chunks in {local_train_dir} -- run Step 1 first."
    for ci, path in enumerate(chunk_files):
        df = pd.read_parquet(path)
        df[time_col] = pd.to_datetime(df[time_col])
        for key, g in df.groupby(group_col, sort=False):
            g = g.sort_values(time_col)
            t = g[time_col].reset_index(drop=True)
            assert (t.diff().dropna() == pd.Timedelta(days=1)).all(), f"{key}: dates are not consecutive"
            sales = g[target_col].to_numpy(dtype=np.float32)
            assert (sales >= 0).all() and np.allclose(sales, np.rint(sales)), f"{key}: non-count target"

            tr = (t <= TRAIN_END).to_numpy()
            va = (t >= VAL_START).to_numpy()
            payload = dict(val_sales=sales[va], val_start=np.array(str(t[va].iloc[0].date())) if va.any() else np.array(""))
            if tr.any():
                payload.update(train_sales=sales[tr], train_start=np.array(str(t[tr].iloc[0].date())))
            np.savez(os.path.join(CACHE_DIR, f"{safe_name(key)}.npz"), **payload)

            row = dict(key=str(key), first_date=t.iloc[0], last_date=t.iloc[-1],
                       n_train=int(tr.sum()), n_val=int(va.sum()),
                       total_sales=float(sales.sum()),
                       recent_sales=float(sales[(t >= RECENT_START).to_numpy()].sum()))
            row.update({c: str(g[c].iloc[0]) for c in static_covariates})
            rows.append(row)
        del df
        gc.collect()
        print(f"  chunk {ci+1}/{len(chunk_files)} -- series so far: {len(rows):,}")
    series_table = pd.DataFrame(rows)
    series_table.to_parquet(series_table_path, index=False)
print(f"Series in cache: {len(series_table):,}")

  chunk 1/12 -- series so far: 3,096
  chunk 2/12 -- series so far: 6,647
  chunk 3/12 -- series so far: 10,229
  chunk 4/12 -- series so far: 13,449
  chunk 5/12 -- series so far: 16,981
  chunk 6/12 -- series so far: 20,001
  chunk 7/12 -- series so far: 23,093
  chunk 8/12 -- series so far: 26,291
  chunk 9/12 -- series so far: 29,169
  chunk 10/12 -- series so far: 31,554
  chunk 11/12 -- series so far: 32,241
  chunk 12/12 -- series so far: 32,244
Series in cache: 32,244


In [36]:
st = series_table
short = st[(st["first_date"] != TRAIN_START) | (st["last_date"] != VAL_END)]
assert short.empty, (
    f"{len(short):,} series do not span {TRAIN_START.date()} -> {VAL_END.date()}, e.g.\n"
    f"{short[['key', 'first_date', 'last_date']].head().to_string(index=False)}")

st["dead"] = st["total_sales"] == 0
st["in_train"] = st["in_val"] = st["in_forecast"] = ~st["dead"]
print(f"series: {len(st):,} | excluded (no sales at all): {int(st.dead.sum()):,} | used: {int((~st.dead).sum()):,}")

series: 32,244 | excluded (no sales at all): 0 | used: 32,244


In [37]:
CODE_MAPS = {c: {v: i for i, v in enumerate(VOCAB[c])} for c in static_covariates}
codes = pd.DataFrame({c: st[c].map(CODE_MAPS[c]).fillna(0).astype(np.int64) for c in static_covariates})
n_unknown = (codes == 0).sum()
print("series with an unknown (code 0) category:", n_unknown[n_unknown > 0].to_dict() or "none")

series_index = pd.concat([st[["key", "in_train", "in_val", "in_forecast"]], codes], axis=1)
series_index.to_parquet(os.path.join(CACHE_DIR, "series_index.parquet"), index=False)


def test_static_codes(codes_df):
    assert list(codes_df.columns) == list(EMBEDDING_SIZES), \
        f"Column order {list(codes_df.columns)} differs from the embedding configuration."
    for c in codes_df.columns:
        v = codes_df[c]
        assert v.notna().all(), f"{c}: blank codes"
        assert np.array_equal(v, np.round(v)), f"{c}: codes must be whole numbers"
        assert v.min() >= 0 and v.max() < EMBEDDING_SIZES[c][0], \
            f"{c}: codes {v.min()}..{v.max()} fall outside table rows 0..{EMBEDDING_SIZES[c][0] - 1}"
    print(f"TEST 1 PASSED -- {len(codes_df):,} series x {codes_df.shape[1]} columns are valid embedding indices.")


test_static_codes(codes)

series with an unknown (code 0) category: none
TEST 1 PASSED -- 32,244 series x 9 columns are valid embedding indices.


In [42]:
class TargetSeq(collections.abc.Sequence):
    def __init__(self, keys, code_rows, split, cache_in_ram=True):
        self.keys, self.code_rows, self.split = list(keys), code_rows, split
        self._ram = {} if cache_in_ram else None

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        if self._ram is not None and i in self._ram:
            return self._ram[i]
        with np.load(os.path.join(CACHE_DIR, f"{safe_name(self.keys[i])}.npz")) as z:
            sales, start = z[f"{self.split}_sales"], str(z[f"{self.split}_start"])
        ts = TimeSeries.from_times_and_values(
            pd.date_range(start, periods=len(sales), freq=FREQ),
            sales.reshape(-1, 1).astype(np.float32),
            columns=[target_col],
            static_covariates=pd.DataFrame([self.code_rows[i]], columns=static_covariates, dtype=np.float32),
        )
        if self._ram is not None:
            self._ram[i] = ts
        return ts


class WeightSeq(collections.abc.Sequence):
    def __init__(self, target_seq):
        self.target_seq = target_seq

    def __len__(self):
        return len(self.target_seq)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        ts = self.target_seq[i]
        w = WEIGHT_BY_DATE.reindex(ts.time_index).to_numpy()
        assert not np.isnan(w).any(), "a series date is missing from the calendar"
        return TimeSeries.from_times_and_values(ts.time_index, w.reshape(-1, 1).astype(np.float32),
                                                columns=["WEIGHT"])


class SharedCov(collections.abc.Sequence):
    def __init__(self, cov, n):
        self.cov, self.n = cov, n

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self.cov for _ in range(*i.indices(self.n))]
        return self.cov


def make_sets(mask_col, split, shuffle_seed=None):
    sel = series_index[series_index[mask_col]]
    if shuffle_seed is not None:
        # Darts reads validation windows in series order and never shuffles them. With
        # limit_val_batches=500 (500 x 256 = 128K windows, 23 per series) only the first ~5.5K
        # series would ever be scored -- the first ~190 dealers by code. A fixed random order
        # makes that subset representative and identical on every run.
        sel = sel.sample(frac=1.0, random_state=shuffle_seed)
    target = TargetSeq(sel["key"], sel[static_covariates].to_numpy(), split)
    return target, SharedCov(SHARED_COV, len(target)), WeightSeq(target)


train_seq, train_cov, train_w = make_sets("in_train", "train")
val_seq,   val_cov,   val_w   = make_sets("in_val",   "val", shuffle_seed=42)
print(f"train series: {len(train_seq):,} | val series: {len(val_seq):,}")

_probe = train_seq[0]
print("Probe series static covariates (codes, as the model receives them):")
print(_probe.static_covariates)
test_static_codes(_probe.static_covariates.reset_index(drop=True))

train series: 32,244 | val series: 32,244
Probe series static covariates (codes, as the model receives them):
static_covariates  PARENT_DEALER_CODE  MODEL_FAMILY  BRAKE_TYPE  \
NET_SALES                        78.0           5.0         2.0   

static_covariates  IGNITION_TYPE  WHEEL_TYPE  COLOUR  X_CITY_CATEGORY  \
NET_SALES                    2.0         2.0     4.0              3.0   

static_covariates  ZONAL_OFFICE_NAME  DEALER_CITY  
NET_SALES                        4.0        667.0  
TEST 1 PASSED -- 1 series x 9 columns are valid embedding indices.


In [43]:
LEARNING_RATE = None        # None = Darts default (1e-3)


def build_model(model_name, n_epochs=100, callbacks=(), limit_train_batches=1250, limit_val_batches=500,
                save_checkpoints=True, use_embeddings=True, logger=None):
    trainer_kwargs = {
        "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
        "devices": 1,
        "callbacks": list(callbacks),
        "gradient_clip_val": 0.1,
        "precision": "bf16-mixed" if torch.cuda.is_available() else "32-true",
        "limit_train_batches": limit_train_batches,
        "limit_val_batches": limit_val_batches,
    }
    if logger is not None:           # None = leave Darts' default; the smoke tests rely on that
        trainer_kwargs["logger"] = logger
    return TFTModel(
        input_chunk_length=INPUT_CHUNK_LENGTH,
        output_chunk_length=OUTPUT_CHUNK_LENGTH,
        hidden_size=32,
        lstm_layers=4,
        num_attention_heads=4,
        dropout=0.05,
        batch_size=256,
        n_epochs=n_epochs,
        likelihood=NegativeBinomialLikelihood(),
        loss_fn=None,
        use_reversible_instance_norm=False,
        # a COPY, so nothing Darts does to its dictionary can touch ours
        categorical_embedding_sizes=dict(EMBEDDING_SIZES) if use_embeddings else None,
        add_relative_index=True,
        optimizer_kwargs={"lr": LEARNING_RATE} if LEARNING_RATE else None,
        random_state=42,
        save_checkpoints=save_checkpoints,
        force_reset=True,
        model_name=model_name,
        pl_trainer_kwargs=trainer_kwargs,
    )


FIT_COMMON = dict(dataloader_kwargs={"num_workers": 0, "pin_memory": torch.cuda.is_available()})

In [ ]:
N_SMOKE = 64
smoke_train = TargetSeq(train_seq.keys[:N_SMOKE], train_seq.code_rows[:N_SMOKE], "train")


def smoke_fit(use_embeddings):
    m = build_model("smoke_embedding_test", n_epochs=1, limit_train_batches=2, limit_val_batches=1,
                    save_checkpoints=False, use_embeddings=use_embeddings)
    m.fit(series=smoke_train, future_covariates=SharedCov(SHARED_COV, len(smoke_train)),
          sample_weight=WeightSeq(smoke_train), verbose=False, **FIT_COMMON)
    return m


# ---- Test 2 ------------------------------------------------------------------
smoke = smoke_fit(use_embeddings=True)
report, n_emb = assert_embeddings(smoke.model, static_covariates, EMBEDDING_SIZES)
print(report.to_string(index=False))
print(f"\nTEST 2 PASSED -- every static column has its own embedding table ({n_emb:,} parameters).")

# ---- Test 3 ------------------------------------------------------------------
tables = smoke.model.input_embeddings.embeddings
for i, c in enumerate(static_covariates):
    W = tables[f"static_covariate_{i}"].weight.detach().float().cpu().numpy()
    if W.shape[0] < 4:          # needs codes 1, 2, 3 (binary columns only have 1 and 2)
        continue
    step_1_to_2, step_2_to_3 = W[2] - W[1], W[3] - W[2]
    on_a_line = np.allclose(step_1_to_2, step_2_to_3, atol=1e-6)
    assert not on_a_line, f"{c}: codes 1, 2, 3 lie on a straight line -- treated as a number"
print("TEST 3 PASSED -- codes are independent learned vectors, not points on a number line.")

# ---- Negative control ---------------------------------------------------------
control = smoke_fit(use_embeddings=False)
print("\nNegative control (no embeddings), as the network sees it:")
print(embedding_report(control.model, static_covariates, EMBEDDING_SIZES)
      [["column", "treated_as", "table_rows"]].to_string(index=False))
try:
    assert_embeddings(control.model, static_covariates, EMBEDDING_SIZES)
    raise AssertionError("Negative control PASSED the check -- the tests cannot detect the problem.")
except RuntimeError:
    print("NEGATIVE CONTROL OK -- the check correctly fails when categories are fed as numbers.")

del smoke, control
gc.collect()

Detected user-defined float16-like precision. For mixed precision training, recommended options are 'bf16-mixed' and '16-mixed'.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.
You are using a CUDA device ('NVIDIA RTX 2000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=1` reached.
Detected user-defined float16-like precision. For mixed precision training, recommended options are 'bf16-mixed' and '16-mixed'.
Using bfloat16 Automatic Mixed Precision (AMP)
G

            column      internal_name treated_as  table_rows  expected_rows  vector_size  expected_vector_size
PARENT_DEALER_CODE static_covariate_0  EMBEDDING        1103           1103           16                    16
      MODEL_FAMILY static_covariate_1  EMBEDDING          14             14            7                     7
        BRAKE_TYPE static_covariate_2  EMBEDDING           3              3            2                     2
     IGNITION_TYPE static_covariate_3  EMBEDDING           3              3            2                     2
        WHEEL_TYPE static_covariate_4  EMBEDDING           5              5            3                     3
            COLOUR static_covariate_5  EMBEDDING          28             28           10                    10
   X_CITY_CATEGORY static_covariate_6  EMBEDDING           4              4            3                     3
 ZONAL_OFFICE_NAME static_covariate_7  EMBEDDING           6              6            4                     4
 

`Trainer.fit` stopped: `max_epochs=1` reached.



Negative control (no embeddings), as the network sees it:
            column     treated_as  table_rows
PARENT_DEALER_CODE NUMBER (wrong)           0
      MODEL_FAMILY NUMBER (wrong)           0
        BRAKE_TYPE NUMBER (wrong)           0
     IGNITION_TYPE NUMBER (wrong)           0
        WHEEL_TYPE NUMBER (wrong)           0
            COLOUR NUMBER (wrong)           0
   X_CITY_CATEGORY NUMBER (wrong)           0
 ZONAL_OFFICE_NAME NUMBER (wrong)           0
       DEALER_CITY NUMBER (wrong)           0
NEGATIVE CONTROL OK -- the check correctly fails when categories are fed as numbers.


83156

In [45]:
TB_DIR = os.path.join(BASE, "tb_logs")
GRAD_LOG_EVERY = 50          # optimizer steps


class TBDiagnostics(Callback):
    """Extra TensorBoard scalars/histograms. Never raises: logging must not kill a long training run."""

    def __init__(self, static_cols, every=GRAD_LOG_EVERY):
        self.static_cols, self.every = static_cols, every

    @staticmethod
    def _writer(trainer):
        return getattr(trainer.logger, "experiment", None)

    def on_before_optimizer_step(self, trainer, pl_module, optimizer):
        # runs before gradient clipping, so this is the raw norm
        try:
            w = self._writer(trainer)
            if w is None or trainer.global_step % self.every:
                return
            norms = [p.grad.detach().float().norm(2) for p in pl_module.parameters() if p.grad is not None]
            if norms:
                w.add_scalar("diag/grad_norm", torch.stack(norms).norm(2).item(), trainer.global_step)
        except Exception as e:
            print("TBDiagnostics grad_norm skipped:", e)

    def on_train_epoch_end(self, trainer, pl_module):
        try:
            w = self._writer(trainer)
            if w is None:
                return
            tables = pl_module.model.input_embeddings.embeddings
            for i, c in enumerate(self.static_cols):
                W = tables[f"static_covariate_{i}"].weight.detach().float().cpu()
                w.add_scalar(f"emb_norm/{c}", W.norm(dim=1).mean().item(), trainer.current_epoch)
                w.add_histogram(f"emb_hist/{c}", W.numpy().ravel(), trainer.current_epoch)
            w.flush()
        except Exception as e:
            print("TBDiagnostics embeddings skipped:", e)


def make_tb_logger(model_name):
    return TensorBoardLogger(save_dir=TB_DIR, name=model_name, version="", default_hp_metric=False)

In [46]:
MODEL_NAME = f"daily_tft_negbin_emb_{datetime.now().strftime('%Y-%m-%d_%H_%M_%S')}"
print("Model name:", MODEL_NAME)

tb_logger = make_tb_logger(MODEL_NAME)
model = build_model(
    MODEL_NAME,
    callbacks=[EarlyStopping(monitor="val_loss", patience=5, min_delta=1e-4, mode="min"),
               EmbeddingGate(static_covariates, EMBEDDING_SIZES),
               LearningRateMonitor(logging_interval="epoch"),
               TBDiagnostics(static_covariates)],
    logger=tb_logger,
)
tb_logger.log_hyperparams(dict(
    input_chunk_length=INPUT_CHUNK_LENGTH, output_chunk_length=OUTPUT_CHUNK_LENGTH,
    hidden_size=32, lstm_layers=4, num_attention_heads=4, dropout=0.05, batch_size=256,
    festive_weight=FESTIVE_WEIGHT, learning_rate=LEARNING_RATE or 1e-3,
    n_train_series=len(train_seq), n_val_series=len(val_seq),
    **{f"emb_dim_{c}": EMBEDDING_SIZES[c][1] for c in static_covariates},
))
model.fit(
    series=train_seq, future_covariates=train_cov, sample_weight=train_w,
    val_series=val_seq, val_future_covariates=val_cov, val_sample_weight=val_w,
    verbose=True, **FIT_COMMON,
)

Model name: daily_tft_negbin_emb_2026-10-09_11_37_06


Detected user-defined float16-like precision. For mixed precision training, recommended options are 'bf16-mixed' and '16-mixed'.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                              | Type                             | Params | Mode 
------------------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0      | train
1  | val_metrics                       | MetricCollection                 | 0      | train
2  | input_embeddings                  | _MultiEmbedding                  | 31.3 K | train
3  | static_covariates_vsn             | _VariableSelectionNetwork        | 1.8 K  | train
4  | encoder_vsn                       | _VariableSelectionNetwork        | 74.3 K | train
5  | decoder_vsn      

EmbeddingGate: PASSED -- 9 static columns use embedding tables (31,321 embedding parameters).


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

... (32 outputs left out)

TBDiagnostics embeddings skipped: '_TFTModule' object has no attribute 'model'


Validation: |          | 0/? [00:00<?, ?it/s]

TBDiagnostics embeddings skipped: '_TFTModule' object has no attribute 'model'


TFTModel(output_chunk_shift=0, hidden_size=32, lstm_layers=4, num_attention_heads=4, full_attention=False, feed_forward=GatedResidualNetwork, dropout=0.05, hidden_continuous_size=8, categorical_embedding_sizes={'PARENT_DEALER_CODE': (1103, 16), 'MODEL_FAMILY': (14, 7), 'BRAKE_TYPE': (3, 2), 'IGNITION_TYPE': (3, 2), 'WHEEL_TYPE': (5, 3), 'COLOUR': (28, 10), 'X_CITY_CATEGORY': (4, 3), 'ZONAL_OFFICE_NAME': (6, 4), 'DEALER_CITY': (827, 16)}, add_relative_index=True, skip_interpolation=False, loss_fn=None, likelihood=NegativeBinomialLikelihood(prior_strength=1.0), norm_type=LayerNorm, use_static_covariates=True, input_chunk_length=365, output_chunk_length=101, batch_size=256, n_epochs=100, use_reversible_instance_norm=False, optimizer_kwargs=None, random_state=42, save_checkpoints=True, force_reset=True, model_name=daily_tft_negbin_emb_2026-10-09_11_37_06, pl_trainer_kwargs={'accelerator': 'gpu', 'devices': 1, 'callbacks': [<pytorch_lightning.callbacks.early_stopping.EarlyStopping object at

In [47]:
best = TFTModel.load_from_checkpoint(MODEL_NAME, best=True, map_location="cpu")
report, n_emb = assert_embeddings(best.model, static_covariates, EMBEDDING_SIZES)
print(report.to_string(index=False))
print(f"Best checkpoint uses embeddings ({n_emb:,} parameters).")
tb_logger.experiment.add_scalar("diag/embedding_parameters", int(n_emb), 0)
tb_logger.experiment.flush()
print("TensorBoard: tensorboard --logdir", TB_DIR)

SHARED_COV.to_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
metadata = dict(
    model_name=MODEL_NAME,
    created=datetime.now().isoformat(timespec="seconds"),
    dates={k: ROLES[k] for k in ["train_start", "train_end", "val_start", "val_end",
                                 "forecast_start", "forecast_end"]},
    input_chunk_length=INPUT_CHUNK_LENGTH, output_chunk_length=OUTPUT_CHUNK_LENGTH,
    embedding_sizes={c: list(v) for c, v in EMBEDDING_SIZES.items()},
    embedding_parameters=int(n_emb),
    festive_weight=FESTIVE_WEIGHT,
    series=dict(train=int(st.in_train.sum()), val=int(st.in_val.sum()), forecast=int(st.in_forecast.sum())),
    versions=dict(darts=darts.__version__, torch=torch.__version__),
)
with open(os.path.join(BASE, "run_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved shared_cov.pkl and run_metadata.json")

            column      internal_name treated_as  table_rows  expected_rows  vector_size  expected_vector_size
PARENT_DEALER_CODE static_covariate_0  EMBEDDING        1103           1103           16                    16
      MODEL_FAMILY static_covariate_1  EMBEDDING          14             14            7                     7
        BRAKE_TYPE static_covariate_2  EMBEDDING           3              3            2                     2
     IGNITION_TYPE static_covariate_3  EMBEDDING           3              3            2                     2
        WHEEL_TYPE static_covariate_4  EMBEDDING           5              5            3                     3
            COLOUR static_covariate_5  EMBEDDING          28             28           10                    10
   X_CITY_CATEGORY static_covariate_6  EMBEDDING           4              4            3                     3
 ZONAL_OFFICE_NAME static_covariate_7  EMBEDDING           6              6            4                     4
 